# ANAC Bandi di Gara — Procedure e spesa della PA italiana (2016-2025)
Dataset: `anac_bandi_gara` (ANAC)

Analisi pubblica: [README](../README.md)

In [1]:
import duckdb
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

FIGS = Path('../figures')
FIGS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

con = duckdb.connect()
con.execute('INSTALL httpfs; LOAD httpfs;')
con.execute("SET s3_region='us-east-1';")
con.execute("SET s3_access_key_id='';")
con.execute("SET s3_secret_access_key='';")
con.execute("SET s3_session_token='';")

GCS_PATH = 'gs://dataciviclab-clean/anac_bandi_gara/*/anac_bandi_gara_*_clean.parquet'
print('DuckDB ready, GCS anonymous access')

DuckDB ready, GCS anonymous access


In [2]:
# 1. Trend decennale 2016-2025
df_trend = con.execute(f"""
    SELECT anno_pubblicazione,
           COUNT(*) AS n_lotti,
           ROUND(SUM(importo_lotto) / 1e9, 1) AS importo_mld,
           ROUND(AVG(importo_lotto), 0) AS importo_medio
    FROM '{GCS_PATH}'
    GROUP BY anno_pubblicazione
    ORDER BY anno_pubblicazione
""").fetchdf()

print('Trend 2016-2025:')
print(df_trend.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Trend 2016-2025:
 anno_pubblicazione  n_lotti  importo_mld  importo_medio
               2016   332115        166.9       502587.0
               2017   374234        214.8       574106.0
               2018   371944        240.1       645483.0
               2019   369654        239.9       648914.0
               2020   388451        251.4       647073.0
               2021   431227        289.9       672175.0
               2022   490443        425.4       867416.0
               2023   655125        404.2       616948.0
               2024  1099596        637.2       579491.0
               2025  1475581        634.7       430166.0


In [3]:
# Figura 1: Trend decennale
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

colors_t = ['#95a5a6'] * 7 + ['#2c3e50'] * 3

bars1 = ax1.bar(df_trend['anno_pubblicazione'].astype(str), df_trend['importo_mld'], color=colors_t, edgecolor='white', width=0.6)
for bar, val in zip(bars1, df_trend['importo_mld']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val:.0f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax1.set_title('Importo totale (miliardi di €)', fontweight='bold')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.set_xticklabels(df_trend['anno_pubblicazione'].astype(str), rotation=45)

bars2 = ax2.bar(df_trend['anno_pubblicazione'].astype(str), df_trend['n_lotti'] / 1e6, color=colors_t, edgecolor='white', width=0.6)
for bar, val in zip(bars2, df_trend['n_lotti']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val/1e6:.2f}M', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax2.set_title('Numero di lotti (milioni)', fontweight='bold')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.set_xticklabels(df_trend['anno_pubblicazione'].astype(str), rotation=45)

fig.suptitle('Bandi di gara ANAC — 2016-2025', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(FIGS / 'appalti_trend_decennale.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura: appalti_trend_decennale.png')

Figura: appalti_trend_decennale.png


In [4]:
# 2. Evoluzione procedure
df_proc = con.execute(f"""
    SELECT anno_pubblicazione,
           CASE
               WHEN tipo_scelta_contraente LIKE 'AFFIDAMENTO DIRETTO%' THEN 'Affidamento diretto'
               WHEN tipo_scelta_contraente LIKE 'PROCEDURA APERTA%' THEN 'Procedura aperta'
               WHEN tipo_scelta_contraente LIKE 'PROCEDURA NEGOZIATA%' THEN 'Procedura negoziata'
               WHEN tipo_scelta_contraente LIKE 'PROCEDURA RISTRETTA%' THEN 'Procedura ristretta'
               WHEN tipo_scelta_contraente LIKE 'AFFIDAMENTO IN ECONOMIA%' THEN 'Affidamento in economia'
               ELSE 'Altre'
           END AS procedura,
           COUNT(*) AS n_lotti,
           ROUND(SUM(importo_lotto) / 1e9, 1) AS importo_mld
    FROM '{GCS_PATH}'
    WHERE tipo_scelta_contraente IS NOT NULL
    GROUP BY anno_pubblicazione, procedura
    ORDER BY anno_pubblicazione, n_lotti DESC
""").fetchdf()

print('Evoluzione procedure (anni selezionati):')
for anno in [2016, 2019, 2022, 2025]:
    sub = df_proc[df_proc['anno_pubblicazione'] == anno]
    print(f'\n--- {anno} ---')
    print(sub[['procedura', 'n_lotti', 'importo_mld']].to_string(index=False))

Evoluzione procedure (anni selezionati):

--- 2016 ---
              procedura  n_lotti  importo_mld
    Affidamento diretto   131205         42.8
    Procedura negoziata    85049         34.5
Affidamento in economia    41800          2.5
       Procedura aperta    39832         52.5
                  Altre    28891         18.5
    Procedura ristretta     5338         16.1

--- 2019 ---
              procedura  n_lotti  importo_mld
    Affidamento diretto   160762         48.2
    Procedura negoziata   110751         48.5
       Procedura aperta    61396        109.3
                  Altre    15623         12.3
Affidamento in economia    13381          1.1
    Procedura ristretta     7741         20.5

--- 2022 ---
              procedura  n_lotti  importo_mld
    Affidamento diretto   298001        121.5
    Procedura negoziata   103574         73.6
       Procedura aperta    63783        189.8
    Procedura ristretta    19376         33.4
                  Altre     5701          7

In [5]:
# Figura 2: Evoluzione procedure stacked
years = [2016, 2019, 2022, 2023, 2024, 2025]
proc_order = ['Affidamento diretto', 'Affidamento in economia', 'Procedura negoziata',
              'Procedura ristretta', 'Procedura aperta', 'Altre']
colors_p = ['#e74c3c', '#95a5a6', '#3498db', '#2c3e50', '#27ae60', '#f39c12']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for metric, ax, ylabel in [('n_lotti', ax1, 'Numero lotti'), ('importo_mld', ax2, 'Miliardi di €')]:
    bottom = np.zeros(len(years))
    for proc, color in zip(proc_order, colors_p):
        vals = []
        for y in years:
            sub = df_proc[(df_proc['anno_pubblicazione'] == y) & (df_proc['procedura'] == proc)]
            vals.append(sub[metric].sum() if len(sub) > 0 else 0)
        ax.bar(range(len(years)), vals, bottom=bottom, label=proc, color=color, edgecolor='white', width=0.6)
        bottom += vals
    ax.set_xticks(range(len(years)))
    ax.set_xticklabels([str(y) for y in years])
    ax.set_ylabel(ylabel)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

ax1.legend(fontsize=9, loc='upper left')
ax1.set_title('Lotti per procedura', fontweight='bold')
ax2.set_title('Valore per procedura', fontweight='bold')
fig.suptitle('Evoluzione delle procedure di scelta del contraente', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGS / 'appalti_evoluzione_procedure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura: appalti_evoluzione_procedure.png')

Figura: appalti_evoluzione_procedure.png


In [6]:
# 3. Distribuzione territoriale 2025
df_terr = con.execute(f"""
    SELECT sezione_regionale,
           COUNT(*) AS n_lotti,
           ROUND(SUM(importo_lotto) / 1e9, 1) AS importo_mld
    FROM '{GCS_PATH}'
    WHERE anno_pubblicazione = 2025 AND sezione_regionale IS NOT NULL
    GROUP BY sezione_regionale
    ORDER BY importo_mld DESC
""").fetchdf()

print('Top 10 sezioni regionali 2025:')
print(df_terr.head(10).to_string(index=False))

Top 10 sezioni regionali 2025:
               sezione_regionale  n_lotti  importo_mld
      SEZIONE REGIONALE CENTRALE   321446        192.9
     SEZIONE REGIONALE LOMBARDIA   187769         76.6
                NON CLASSIFICATO    72678         55.9
      SEZIONE REGIONALE CAMPANIA    69756         44.6
       SEZIONE REGIONALE SICILIA    87386         41.4
       SEZIONE REGIONALE TOSCANA    65590         35.4
         SEZIONE REGIONALE LAZIO    69641         33.2
        SEZIONE REGIONALE VENETO    88905         30.6
      SEZIONE REGIONALE PIEMONTE    84692         22.2
SEZIONE REGIONALE EMILIA ROMAGNA    76309         20.8


In [7]:
# Figura 3: Distribuzione territoriale
df_plot = df_terr.copy()
df_plot['nome_corto'] = df_plot['sezione_regionale'].str.replace('SEZIONE REGIONALE ', '')

fig, ax = plt.subplots(figsize=(12, 6))
med = df_plot['importo_mld'].median()
colors_r = ['#e74c3c' if v >= med else '#95a5a6' for v in df_plot['importo_mld']]

bars = ax.barh(df_plot['nome_corto'][::-1], df_plot['importo_mld'][::-1], color=colors_r[::-1], edgecolor='white')
ax.set_xlabel('Miliardi di €')
ax.set_title('Bandi di gara per sezione regionale ANAC — 2025', fontweight='bold', fontsize=14)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for bar, val in zip(bars, df_plot['importo_mld'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f} mld', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIGS / 'appalti_distribuzione_territoriale.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura: appalti_distribuzione_territoriale.png')

Figura: appalti_distribuzione_territoriale.png


In [8]:
# 4. Categorie 2025
df_cat = con.execute(f"""
    SELECT oggetto_principale_contratto,
           COUNT(*) AS n_lotti,
           ROUND(SUM(importo_lotto) / 1e9, 1) AS importo_mld,
           ROUND(AVG(importo_lotto), 0) AS importo_medio
    FROM '{GCS_PATH}'
    WHERE anno_pubblicazione = 2025 AND oggetto_principale_contratto IS NOT NULL
    GROUP BY oggetto_principale_contratto
    ORDER BY importo_mld DESC
""").fetchdf()

print('Categorie 2025:')
print(df_cat.to_string(index=False))

Categorie 2025:
oggetto_principale_contratto  n_lotti  importo_mld  importo_medio
                   FORNITURE   649450        287.1       442047.0
                     SERVIZI   658564        267.5       406154.0
                      LAVORI   167566         80.2       478490.0


In [9]:
# Figura 4: Categorie
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

colors_c = ['#3498db', '#2c3e50', '#27ae60']
labels_c = df_cat['oggetto_principale_contratto'].tolist()

bars1 = ax1.bar(labels_c, df_cat['importo_mld'], color=colors_c, edgecolor='white', width=0.5)
for bar, val in zip(bars1, df_cat['importo_mld']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{val:.0f} mld', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_title('Importo (miliardi di €)', fontweight='bold')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

wedges, texts, autotexts = ax2.pie(df_cat['n_lotti'], labels=labels_c, autopct='%1.0f%%',
    colors=colors_c, startangle=90, textprops={'fontsize': 10})
ax2.set_title('Lotti (% sul totale)', fontweight='bold')

fig.suptitle('Categorie merceologiche — 2025', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGS / 'appalti_categorie.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura: appalti_categorie.png')

Figura: appalti_categorie.png


In [10]:
# Riepilogo finale
totali = con.execute(f"""
    SELECT COUNT(*) AS tot_lotti,
           ROUND(SUM(importo_lotto) / 1e9, 1) AS tot_mld,
           ROUND(AVG(importo_lotto), 0) AS media_generale,
           MIN(anno_pubblicazione) AS da,
           MAX(anno_pubblicazione) AS a
    FROM '{GCS_PATH}'
""").fetchdf()

print('RIEPILOGO — ANAC 2016-2025')
print(f"Lotti totali: {int(totali['tot_lotti'].values[0]):,}")
print(f"Importo totale: {totali['tot_mld'].values[0]:.0f} miliardi")
print(f"Media per lotto: {totali['media_generale'].values[0]:,.0f} €")

crescita = df_trend[df_trend['anno_pubblicazione']==2025]['n_lotti'].values[0] / df_trend[df_trend['anno_pubblicazione']==2016]['n_lotti'].values[0]
print(f"Crescita lotti 2016→2025: {crescita:.1f}x")

RIEPILOGO — ANAC 2016-2025
Lotti totali: 5,988,370
Importo totale: 3504 miliardi
Media per lotto: 585,215 €
Crescita lotti 2016→2025: 4.4x


In [11]:
print('Tutte le figure generate:')
for f in sorted(FIGS.glob('appalti_*.png')):
    print(f'  ✓ {f.name}')

Tutte le figure generate:
  ✓ appalti_categorie.png
  ✓ appalti_confronto_procedure.png
  ✓ appalti_distribuzione_territoriale.png
  ✓ appalti_evoluzione_procedure.png
  ✓ appalti_trend_decennale.png
